In [ ]:
"""
Stage-2 ABC local refinement for Baseline 0 single-antigen CAR-T model.

Purpose:
    Stage 1 used broad prior sampling and found a very narrow retained region.
    Stage 2 samples locally around retained Stage-1 parameter sets to enlarge
    the retained ensemble before dual-antigen 2s/dCAR/2v comparison.

Required files:
    1. baseline0_single_antigen.py
    2. sampled_parameter_sets CSV from Stage 1
    3. validation_summary CSV from Stage 1

Example using the original 4 retained sets from the 2000 run:

    python stage2_abc_local_refinement.py \
        --params_csv "sampled_parameter_sets (1).csv" \
        --summary_csv "validation_summary (1).csv" \
        --seed_param_ids "7,591,1375,1412" \
        --n_parameter_sets 3000 \
        --cohort_size 50

Example using all retained sets from the 5000 run:

    python stage2_abc_local_refinement.py \
        --params_csv "sampled_parameter_sets (2)(1).csv" \
        --summary_csv "validation_summary (2)(1).csv" \
        --use_all_retained \
        --n_parameter_sets 5000 \
        --cohort_size 50
"""
%run "./baseline0-single-antigen.ipynb"

from __future__ import annotations

import argparse
from dataclasses import asdict, fields
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from baseline0_single_antigen import (
    ModelParams,
    simulate_cohort,
    cohort_summary,
    validation_gates,
    calibration_distance,
)


# ============================================================
# 1. Prior bounds from Stage-1 Baseline 0
# ============================================================

PARAM_BOUNDS = {
    "E0": (5e7, 2e8),
    "effector_doubling_time_days": (1.3, 2.8),
    "base_death_rate_per_day": (0.01, 0.06),
    "aicd_max_rate_per_day": (0.20, 0.70),
    "aicd_mid_day": (8.0, 15.0),
    "aicd_width_days": (1.5, 5.0),
    "stim_half_saturation_cells": (1e6, 1e9),
    "max_kills_per_effector_per_day": (0.05, 5.0),
    "kill_threshold_molecules": (1e3, 3e3),
    "kill_steepness": (2.0, 8.0),
    "tumour_growth_rate_per_day": (0.0, 0.035),
    "effector_noise_sigma": (0.01, 0.08),
    "kill_noise_sigma": (0.05, 0.25),
}

# Parameters that are more naturally perturbed multiplicatively.
LOG_SCALE_PARAMS = {
    "E0",
    "stim_half_saturation_cells",
    "max_kills_per_effector_per_day",
    "kill_threshold_molecules",
}

# Fixed simulation settings.
FIXED_PARAMS = {
    "t_end_days",
    "dt_days",
    "clearance_threshold_cells",
}


# ============================================================
# 2. Helper functions
# ============================================================

def parse_param_ids(text: str) -> list[int]:
    """Parse comma-separated param IDs."""
    if not text:
        return []
    return [int(x.strip()) for x in text.split(",") if x.strip()]


def clip_value(x: float, low: float, high: float) -> float:
    """Clip a numeric value to allowed prior bounds."""
    return float(min(max(x, low), high))


def row_to_model_params(row: pd.Series) -> ModelParams:
    """Reconstruct ModelParams from one row of sampled_parameter_sets.csv."""
    valid_names = {f.name for f in fields(ModelParams)}
    kwargs = {}

    for name in valid_names:
        if name in row.index:
            kwargs[name] = float(row[name])

    return ModelParams(**kwargs)


def perturb_one_param(
    rng: np.random.Generator,
    name: str,
    value: float,
    local_width: float,
) -> float:
    """
    Locally perturb one parameter around a retained Stage-1 value.

    local_width controls how local the Stage-2 proposal is:
        0.10 = very local
        0.20 = moderate
        0.35 = broader local refinement
    """

    if name in FIXED_PARAMS:
        return value

    low, high = PARAM_BOUNDS[name]

    if name in LOG_SCALE_PARAMS:
        # Multiplicative local perturbation in log space.
        # local_width is roughly the standard deviation in natural log space.
        proposed = value * rng.lognormal(mean=0.0, sigma=local_width)
        return clip_value(proposed, low, high)

    # Additive perturbation as a fraction of the prior range.
    prior_range = high - low
    proposed = value + rng.normal(loc=0.0, scale=local_width * prior_range)
    return clip_value(proposed, low, high)


def perturb_params(
    rng: np.random.Generator,
    parent: ModelParams,
    local_width: float,
) -> ModelParams:
    """Create a Stage-2 parameter proposal around one retained parent set."""

    parent_dict = asdict(parent)
    new_dict = {}

    for name, value in parent_dict.items():
        if name in PARAM_BOUNDS:
            new_dict[name] = perturb_one_param(
                rng=rng,
                name=name,
                value=float(value),
                local_width=local_width,
            )
        else:
            # Keep simulation constants unchanged.
            new_dict[name] = float(value)

    return ModelParams(**new_dict)


def choose_parent_ids(
    params_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    seed_param_ids: list[int],
    use_all_retained: bool,
) -> list[int]:
    """Choose which retained parameter sets are used as Stage-2 parents."""

    if use_all_retained:
        parent_ids = summary_df.loc[summary_df["retained"], "param_id"].astype(int).tolist()
    else:
        parent_ids = seed_param_ids

    if not parent_ids:
        raise ValueError(
            "No parent parameter IDs provided. Use --seed_param_ids or --use_all_retained."
        )

    available_ids = set(params_df["param_id"].astype(int).tolist())
    missing = [pid for pid in parent_ids if pid not in available_ids]
    if missing:
        raise ValueError(f"These parent param_id values are missing from params_csv: {missing}")

    return parent_ids


# ============================================================
# 3. Stage-2 ABC loop
# ============================================================

def stage2_abc_refinement(
    params_csv: str,
    summary_csv: str,
    seed_param_ids: list[int],
    use_all_retained: bool = False,
    n_parameter_sets: int = 3000,
    cohort_size: int = 50,
    local_width: float = 0.20,
    seed: int = 2027,
    outdir: str = "stage2_outputs",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run Stage-2 ABC local refinement.

    Returns:
        stage2_params_df
        stage2_summary_df
    """

    outdir_path = Path(outdir)
    outdir_path.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(seed)

    params_df = pd.read_csv(params_csv)
    summary_df = pd.read_csv(summary_csv)

    parent_ids = choose_parent_ids(
        params_df=params_df,
        summary_df=summary_df,
        seed_param_ids=seed_param_ids,
        use_all_retained=use_all_retained,
    )

    parent_rows = (
        params_df[params_df["param_id"].isin(parent_ids)]
        .copy()
        .sort_values("param_id")
    )

    parents = {
        int(row["param_id"]): row_to_model_params(row)
        for _, row in parent_rows.iterrows()
    }

    print("\nStage-2 ABC local refinement")
    print("============================")
    print(f"Number of Stage-1 parent sets: {len(parents)}")
    print(f"Parent param_id values: {list(parents.keys())}")
    print(f"Stage-2 proposals: {n_parameter_sets}")
    print(f"Cohort size per proposal: {cohort_size}")
    print(f"Local width: {local_width}")
    print(f"Output directory: {outdir_path.resolve()}\n")

    stage2_param_rows = []
    stage2_summary_rows = []

    parent_id_list = list(parents.keys())

    for stage2_id in range(n_parameter_sets):
        parent_id = int(rng.choice(parent_id_list))
        parent_params = parents[parent_id]

        proposed_params = perturb_params(
            rng=rng,
            parent=parent_params,
            local_width=local_width,
        )

        cohort_df, _ = simulate_cohort(
            params=proposed_params,
            cohort_size=cohort_size,
            rng=rng,
        )

        summary = cohort_summary(cohort_df)
        gates = validation_gates(summary)
        distance = calibration_distance(summary)

        param_row = asdict(proposed_params)
        param_row["stage2_param_id"] = stage2_id
        param_row["parent_param_id"] = parent_id

        summary_row = {
            "stage2_param_id": stage2_id,
            "parent_param_id": parent_id,
            **summary,
            **gates,
            "distance": distance,
        }

        stage2_param_rows.append(param_row)
        stage2_summary_rows.append(summary_row)

        if (stage2_id + 1) % 100 == 0:
            retained_so_far = sum(row["retained"] for row in stage2_summary_rows)
            print(
                f"Sampled {stage2_id + 1}/{n_parameter_sets}; "
                f"retained so far: {retained_so_far}"
            )

    stage2_params_df = pd.DataFrame(stage2_param_rows)
    stage2_summary_df = pd.DataFrame(stage2_summary_rows)

    stage2_params_df.to_csv(outdir_path / "stage2_sampled_parameter_sets.csv", index=False)
    stage2_summary_df.to_csv(outdir_path / "stage2_validation_summary.csv", index=False)

    retained_df = stage2_summary_df[stage2_summary_df["retained"]].copy()
    retained_params_df = stage2_params_df[
        stage2_params_df["stage2_param_id"].isin(retained_df["stage2_param_id"])
    ].copy()

    retained_df.to_csv(outdir_path / "stage2_retained_summary.csv", index=False)
    retained_params_df.to_csv(outdir_path / "stage2_retained_parameter_sets.csv", index=False)

    return stage2_params_df, stage2_summary_df


# ============================================================
# 4. Diagnostics and plots
# ============================================================

def print_stage2_report(summary_df: pd.DataFrame) -> None:
    """Print retained count, gate pass rates, and near-miss diagnostics."""

    gate_cols = [c for c in summary_df.columns if c.startswith("gate_")]
    all_gate_cols = gate_cols + ["retained"]

    n_sampled = len(summary_df)
    n_retained = int(summary_df["retained"].sum())

    print("\nStage-2 results")
    print("===============")
    print(f"Sampled parameter sets:  {n_sampled}")
    print(f"Retained parameter sets: {n_retained}")
    print(f"Retention rate:          {n_retained / max(n_sampled, 1):.4f}")

    print("\nGate pass rates:")
    print(summary_df[all_gate_cols].mean().sort_values(ascending=False))

    summary_df = summary_df.copy()
    summary_df["n_gates_pass"] = summary_df[gate_cols].sum(axis=1)

    near_miss_df = summary_df[
        (summary_df["n_gates_pass"] == len(gate_cols) - 1)
        & (~summary_df["retained"])
    ]

    print(f"\nNear-miss parameter sets passing {len(gate_cols)-1}/{len(gate_cols)} gates: {len(near_miss_df)}")

    if len(near_miss_df) > 0:
        failed_gate_counts = {
            gate: int((near_miss_df[gate] == False).sum())
            for gate in gate_cols
        }
        print("Failed gate among near-misses:")
        for gate, count in sorted(failed_gate_counts.items(), key=lambda x: -x[1]):
            if count > 0:
                print(f"  {gate}: {count}")

    print("\nTop 10 closest Stage-2 parameter sets:")
    display_cols = [
        "stage2_param_id",
        "parent_param_id",
        "distance",
        "t_peak_median",
        "E_peak_median",
        "ORR_rate",
        "CR_rate",
        "escape_rate_among_ORR",
        "ET_gradient",
        "retained",
    ]
    print(summary_df.sort_values("distance").head(10)[display_cols])


def plot_stage2_gate_pass_rates(summary_df: pd.DataFrame, outdir: str) -> None:
    """Plot Stage-2 validation gate pass rates."""

    outdir_path = Path(outdir)
    gate_cols = [c for c in summary_df.columns if c.startswith("gate_")] + ["retained"]

    rates = summary_df[gate_cols].mean().sort_values(ascending=False)

    plt.figure(figsize=(10, 5))
    plt.bar(rates.index, rates.values)
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Pass rate across Stage-2 parameter sets")
    plt.title("Stage-2 validation gate pass rates")
    plt.tight_layout()
    plt.savefig(outdir_path / "stage2_validation_gate_pass_rates.png", dpi=200)
    plt.close()

    rates.rename("pass_rate").to_csv(outdir_path / "stage2_gate_pass_rates.csv")


def plot_retention_by_parent(summary_df: pd.DataFrame, outdir: str) -> None:
    """Plot how many Stage-2 retained proposals came from each parent."""

    outdir_path = Path(outdir)

    by_parent = (
        summary_df
        .groupby("parent_param_id")["retained"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"count": "n_proposals", "sum": "n_retained", "mean": "retention_rate"})
        .reset_index()
    )

    by_parent.to_csv(outdir_path / "stage2_retention_by_parent.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.bar(by_parent["parent_param_id"].astype(str), by_parent["retention_rate"])
    plt.xlabel("Stage-1 parent param_id")
    plt.ylabel("Stage-2 retention rate")
    plt.title("Stage-2 retention rate by parent parameter set")
    plt.tight_layout()
    plt.savefig(outdir_path / "stage2_retention_by_parent.png", dpi=200)
    plt.close()


# ============================================================
# 5. Command-line interface
# ============================================================

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Stage-2 ABC local refinement around retained Stage-1 parameter sets."
    )

    parser.add_argument(
        "--params_csv",
        type=str,
        required=True,
        help="Path to Stage-1 sampled_parameter_sets.csv.",
    )

    parser.add_argument(
        "--summary_csv",
        type=str,
        required=True,
        help="Path to Stage-1 validation_summary.csv.",
    )

    parser.add_argument(
        "--seed_param_ids",
        type=str,
        default="7,591,1375,1412",
        help="Comma-separated Stage-1 retained param_id values used as local parents.",
    )

    parser.add_argument(
        "--use_all_retained",
        action="store_true",
        help="Use all retained parameter sets in summary_csv as Stage-2 parents.",
    )

    parser.add_argument(
        "--n_parameter_sets",
        type=int,
        default=3000,
        help="Number of Stage-2 local proposals.",
    )

    parser.add_argument(
        "--cohort_size",
        type=int,
        default=50,
        help="Virtual cohort size per Stage-2 proposal.",
    )

    parser.add_argument(
        "--local_width",
        type=float,
        default=0.20,
        help="Local perturbation width. Try 0.10, 0.20, or 0.35.",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=2027,
        help="Random seed.",
    )

    parser.add_argument(
        "--outdir",
        type=str,
        default="stage2_outputs",
        help="Output directory.",
    )

    args = parser.parse_args()

    seed_param_ids = parse_param_ids(args.seed_param_ids)

    _, stage2_summary_df = stage2_abc_refinement(
        params_csv=args.params_csv,
        summary_csv=args.summary_csv,
        seed_param_ids=seed_param_ids,
        use_all_retained=args.use_all_retained,
        n_parameter_sets=args.n_parameter_sets,
        cohort_size=args.cohort_size,
        local_width=args.local_width,
        seed=args.seed,
        outdir=args.outdir,
    )

    print_stage2_report(stage2_summary_df)
    plot_stage2_gate_pass_rates(stage2_summary_df, args.outdir)
    plot_retention_by_parent(stage2_summary_df, args.outdir)

    print(f"\nStage-2 outputs written to: {Path(args.outdir).resolve()}")


if __name__ == "__main__":
    main()